In [ ]:
import pandas as pd
import re
import string
from io import StringIO
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import numpy as np
import joblib
import optuna

# Setup NLTK stopwords
try:
    nltk.download('stopwords', quiet=True)
    stop_words_id = set(stopwords.words('indonesian'))
    stop_words_en = set(stopwords.words('english'))
    stop_words = stop_words_id.union(stop_words_en)
except Exception as e:
    print("NLTK stopwords gagal di-download. Menggunakan manual list.")
    stop_words = {
        'dan', 'di', 'yang', 'untuk', 'dari', 'ke', 'pada', 'ini', 'itu', 
        'adalah', 'dengan', 'sebagai', 'oleh', 'akan', 'atau', 'tetapi', 
        'karena', 'jika', 'sementara', 'seperti', 'the', 'a', 'an', 'to', 
        'in', 'on', 'at', 'is', 'are', 'was', 'were', 'be', 'being', 'been'
    }

# Dictionary for typo/slang corrections (combined from train and test data)
typo_corrections = {
    'tau': 'tahu', 'ga': 'tidak', 'ngirip': 'menginap', 'gokil': 'gokil',
    'ngelakuin': 'melakukan', 'bember': 'bumper', 'fabricasi': 'fabrikasi',
    'kasteman': 'custom', 'spesia': 'spesial', 'kenal': 'kena', 'tong olan': 'tambahan',
    'foglem': 'fog lamp', 'menggauza': 'mengganti', 'kecaya': 'kece', 'emak': 'memang',
    'distirk': 'distrik', 'pretah': 'pret', 'kerasa': 'terasa', 'penghinder': 'penghindaran',
    'permasuk': 'memasukkan', 'kurosakan': 'kerusakan', 'komplikat': 'komplit',
    'denang': 'dengan', 'timboi': 'timbo', 'licin': 'licin', 'merundukan': 'merunduk',
    'kempad': 'kepad', 'stadia': 'stadi', 'tertar': 'tertarik', 'yui': 'yoi',
    'bernaval': 'bernavigasi', 'pukuran': 'pukulan', 'kula': 'bola', 'batel': 'badminton',
    'temok': 'tembok', 'memfaal': 'memfamiliarisasi', 'kancini': 'konsistensi',
    'cilatkan': 'latihkan', 'kongga': 'kanggo', 'tentar': 'tentu', 'berfit': 'fitbit',
    'latian': 'latihan', 'repetisi': 'repetisi', 'menunik': 'menu', 'cahat': 'catat',
    'hyper trophy': 'hypertrophy', 'beribadi': 'berbadan', 'apetama': 'pertama',
    'toba': 'coba', 'rupti': 'rupanya', 'dipsepak': 'disepak', 'berguru': 'belajar',
    'masal ap': 'muscle up', 'tepari': 'terapi', 'naret': 'narik', 'pudau': 'pull down',
    'injek': 'injak', 'pecep': 'cepat', 'spes': 'space', 'labor': 'elaborasi',
    'ngakat': 'mengangkat', 'pangga': 'panggil', 'arain': 'tarik', 'petalkan': 'otot kaki',
    'kamar': 'kamera', 'gaba': 'gambar', 'silaturah mi': 'silaturahmi', 'ganti-aki': 'ganti aki',
    'berpamitan': 'berpamitan', 'bodi wix': 'body weight', 'dital': 'ditarik',
    'glut beris': 'glute bridge', 'hanyok': 'hantuk', 'calf resist': 'calf raise',
    'tiba-tiba': 'tiba-tiba sekali', 'war oh': 'workout', 'lapan': 'lapangan',
    'OST-608': 'assisted pull-up', 'glandungan': 'gantungan', 'skap': 'scapular',
    'belau anang': 'berlawanan', 'unitoralis': 'universalis', 'enelurut': 'menurut',
    'dorasi': 'durasi', 'sumba': 'sumber', 'luma': 'lama', 'nakamru': 'makan',
    'dihidayah': 'dehidrasi', 'jaga': 'gagal', 'belifariasi': 'variasi', 'lesh': 'less',
    'sebesar': 'sebisa', 'hanyak': 'hanya', 'terlohor': 'telur', 'tempi': 'tempe',
    'makhlotis': 'makronutrien', 'balen': 'balik', 'panam': 'punggung', 'kesi imbakan': 'keseimbangan',
    'sita': 'sit-up', 'rancel': 'ransel', 'biasab': 'biceps', 'jatoh': 'jatuh',
    'sekolah': 'sekali', 'bantalan': 'bantalan bulat', 'romaine': 'roman', 'nge-strage': 'nge-stretch',
    'penunggam': 'punggung', 'ke egini': 'begini', 'bayset': 'biceps', 'gavis': 'gampang',
    'glamir-glamir': 'lemak-lemak', 'alara': 'olahraga', 'galain': 'gila', 'ngajem': 'nge-gym',
    'beringgui': 'beringas', 'richards': 'recharge', 'gok': 'gokil', 'mumpu': 'mumpuni',
    'kelampunya': 'lampunya', 'lay-bots': 'lay-ups', 'osmosan': 'sesak napas',
    'reen': 'ingin', 'jogin': 'jogging', 'jemerfuloh': 'sempurna', 'talis': 'tali',
    'patulari': 'patu lari', 'robong': 'lobang', 'mencakram': 'mengikat', 'perdam': 'perdam',
    'keling': 'keliling', 'peplafonnya': 'plafonnya', 'ngurangi': 'mengurangi', 'diikat': 'diikat',
    'otor s': 'otorisasi', 'koplingnya': 'kopling', 'tinggalkan': 'gunakan', 'bener-bener': 'benar-benar',
    'nyalan': 'nyalakan', 'in jap': 'injak', 'kematchetan': 'kecemasan', 'panyolis': 'panelis',
    'kentasiknya': 'fantastiknya', 'ketil': 'kecil', 'tampal': 'tampilan', 'bat-paks': 'batas',
    'plakson': 'klakson', 'kanyari-nyari': 'mencari-cari', 'jentat': 'gentar', 'hasar': 'hazard',
    'il as': 'hilang', 'pisa': 'pisah', 'peranya': 'pernah', 'kaunya': 'kamera', 'layak': 'layar',
    'seber-seberin': 'spek-spek', 'liras': 'lisensi', 'transyan': 'transision', 'grub': 'grup',
    'seteranya': 'storenya', 'kotus': 'kualitas', 'kukas': 'kulkas', 'memunuh': 'memenuhi',
    'killer': 'chiller', 'bawa-bahan': 'bahan-bahan', 'perluasan': 'ekspansi', 'peningin': 'pendingin',
    'di odorizer': 'deodorizer', 'sendap': 'bau', 'diffros': 'defrost', 'baska': 'basket',
    'pernahinannya': 'pemanasannya', 'buyang': 'buyar', 'halangan': 'haid', 'membalut': 'membelud',
    'lockmingsnya': 'locking-nya', 'menikwat': 'mengikat'
}

# Fungsi untuk membersihkan teks
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Hapus pola aneh
    text = re.sub(r'^(True|False)(,(True|False))*,,?', '', text)
    text = re.sub(r',,(True|False)(,(True|False))*$', '', text)
    # Hapus emoji dan non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # Koreksi typo
    typo_corrections_local = typo_corrections.copy()
    for slang, formal in typo_corrections_local.items():
        pattern = r'\b' + re.escape(slang) + r'\b'
        text = re.sub(pattern, formal, text, flags=re.IGNORECASE)
    # Case folding
    text = text.lower()
    # Hapus URL, email, angka
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    # Hapus tanda baca kecuali . ! ? ,
    keep_punct = ['!', '?', '.', ',']
    translator = str.maketrans('', '', ''.join([c for c in string.punctuation if c not in keep_punct]))
    text = text.translate(translator)
    # Kurangi huruf berulang
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # Normalisasi spasi
    text = re.sub(r'\s+', ' ', text).strip()
    # Hapus stopwords
    words = [word for word in text.split() if word not in stop_words and len(word) > 2]
    text = ' '.join(words)
    return text.strip()

# Load dataset
file_path = "/kaggle/input/text-bdc/hasil_preprocessing.csv"
df = pd.read_csv(file_path)
print(f"File loaded successfully. Shape: {df.shape}")

# Validasi kolom
if 'text_transcript' not in df.columns:
    text_col = df.select_dtypes(include=['object']).columns[0]
    df = df.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# Hapus baris dengan text_transcript kosong
df.dropna(subset=['text_transcript'], inplace=True)

# Encode label
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'])
print(f"Classes: {le.classes_.tolist()}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text_transcript'],
    df['label'],
    test_size=0.27,
    stratify=df['label'],
    random_state=42
)

# Buat DataFrame untuk train dan test
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

# Preprocessing teks
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# Hapus baris dengan teks kosong setelah cleaning
train_df = train_df[train_df['cleaned_text'] != ""].reset_index(drop=True)
test_df = test_df[test_df['cleaned_text'] != ""].reset_index(drop=True)
print(f"\nTrain: {len(train_df)}, Test: {len(test_df)}")

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    max_features=30000,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm='l2',
    lowercase=False,
    stop_words=None
)

X_train_vec = vectorizer.fit_transform(train_df['cleaned_text'])
X_test_vec = vectorizer.transform(test_df['cleaned_text'])
print(f"TF-IDF Train shape: {X_train_vec.shape}")
print(f"TF-IDF Test shape: {X_test_vec.shape}")

# Modelling dengan Logistic Regression
best_C = 0.5010970952098729
best_penalty = 'l2'
model = LogisticRegression(
    C=best_C,
    penalty=best_penalty,
    solver='lbfgs',
    max_iter=5000,
    class_weight='balanced',
    random_state=42
)

# Latih model
model.fit(X_train_vec, train_df['label'])

# Prediksi pada data test
y_pred = model.predict(X_test_vec)

# Evaluasi model
print("\n=== 📊 EVALUASI MODEL (HYPERPARAMETER OPTUNA TRIAL #46) ===")
print("Confusion Matrix:")
print(confusion_matrix(test_df['label'], y_pred))
print("\nClassification Report:")
print(classification_report(test_df['label'], y_pred, target_names=le.classes_))
f1_macro = f1_score(test_df['label'], y_pred, average='macro')
print(f"\n🎯 F1-MACRO SCORE: {f1_macro:.5f}")

# Simpan model, vectorizer, dan label encoder
joblib.dump(model, 'logreg_optuna_trial46.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("\n✅ Model, vectorizer, dan label encoder berhasil disimpan!")

In [ ]:
import pandas as pd
import joblib

# === 1. Load dataset baru ===
file_path_test = "/kaggle/input/test-text-bismillah/hasil_preprocessing_test_whisper_sma.csv"
df_new = pd.read_csv(file_path_test)
print(f"File test loaded. Shape: {df_new.shape}")

# Pastikan ada kolom teks
if 'text_transcript' not in df_new.columns:
    text_col = df_new.select_dtypes(include=['object']).columns[0]
    df_new = df_new.rename(columns={text_col: 'text_transcript'})
    print(f"Renamed column to 'text_transcript'.")

# === 2. Cleaning text (pakai fungsi clean_text dari code training sebelumnya) ===
df_new['cleaned_text'] = df_new['text_transcript'].apply(clean_text)

# Jangan buang row kosong → isi dengan "0"
df_new['cleaned_text'] = df_new['cleaned_text'].apply(lambda x: x if x != "" else "0")

# === 3. Load artefak training (model, vectorizer, encoder) ===
model = joblib.load("/kaggle/working/logreg_optuna_trial46.pkl")
vectorizer = joblib.load("/kaggle/working/tfidf_vectorizer.pkl")
le = joblib.load("/kaggle/working/label_encoder.pkl")

# === 4. Transform teks baru ke TF-IDF ===
X_new_vec = vectorizer.transform(df_new['cleaned_text'])

# === 5. Prediksi emotion ===
y_pred = model.predict(X_new_vec)
y_pred_label = le.inverse_transform(y_pred)

# === 6. Mapping label -> angka (sesuai permintaan) ===
mapping = {
    "Proud": 0,
    "Trust": 1,
    "Joy": 2,
    "Surprise": 3,
    "Neutral": 4,
    "Sadness": 5,
    "Fear": 6,
    "Anger": 7
}

# Buat dataframe hasil prediksi
df_result = pd.DataFrame({
    "id": df_new.index + 1 if "id" not in df_new.columns else df_new["id"],
    "predicted": [mapping.get(lbl, -1) for lbl in y_pred_label]  # -1 kalau label tidak ada di mapping
})

# Urutkan berdasarkan id
df_result = df_result.sort_values(by="id").reset_index(drop=True)

print("\n=== Hasil Prediksi (contoh 10 baris) ===")
print(df_result.head(10))

# === 7. Simpan hasil ke CSV (submission) ===
df_result.to_csv("submission_34.csv", index=False)
print("\n✅ File submission berhasil disimpan ke 'submission.csv'")

In [ ]:
# === 8. BANDINGKAN DENGAN SUBMISSION LAMA ===
submission_lama_path = "/kaggle/input/submit-waktu-itu/submissionSD2025040000155.csv"

try:
    # Load submission lama
    df_lama = pd.read_csv(submission_lama_path)
    print(f"\nSubmission lama loaded: {submission_lama_path}")
    print(f"Shape: {df_lama.shape}")

    # Pastikan kolom dan urutan id sama
    if 'id' not in df_lama.columns:
        df_lama['id'] = range(1, len(df_lama) + 1)
    df_lama = df_lama[['id', 'predicted']].sort_values('id').reset_index(drop=True)

    # Pastikan df_result juga sudah terurut
    df_baru = df_result.copy()
    df_baru = df_baru[['id', 'predicted']].sort_values('id').reset_index(drop=True)

    # Validasi jumlah baris
    if len(df_lama) != len(df_baru):
        print(f"Warning: Jumlah baris tidak sama! Lama: {len(df_lama)}, Baru: {len(df_baru)}")
    else:
        print(f"Jumlah baris cocok: {len(df_lama)}")

    # Hitung perbedaan
    df_compare = df_lama.copy()
    df_compare['predicted_baru'] = df_baru['predicted'].values
    df_compare['berubah'] = df_compare['predicted'] != df_compare['predicted_baru']

    total_berubah = df_compare['berubah'].sum()
    persentase_berubah = (total_berubah / len(df_compare)) * 100

    print(f"\n=== PERBANDINGAN HASIL ===")
    print(f"Total baris: {len(df_compare)}")
    print(f"Berubah: {total_berubah} baris")
    print(f"Tetap sama: {len(df_compare) - total_berubah} baris")
    print(f"Persentase berubah: {persentase_berubah:.2f}%")

    # Tampilkan contoh yang berubah (maks 10)
    df_berubah = df_compare[df_compare['berubah']]
    if len(df_berubah) > 0:
        print(f"\nContoh {min(10, len(df_berubah))} baris yang BERUBAH:")
        contoh = df_berubah[['id', 'predicted', 'predicted_baru']].head(10)
        # Konversi angka ke label untuk lebih mudah dibaca
        reverse_mapping = {v: k for k, v in mapping.items()}
        contoh['lama_label'] = contoh['predicted'].map(reverse_mapping)
        contoh['baru_label'] = contoh['predicted_baru'].map(reverse_mapping)
        print(contoh[['id', 'lama_label', 'baru_label']])
    else:
        print("\nTidak ada perubahan!")

    # Simpan diff (opsional)
    diff_path = "diff_vs_lama.csv"
    df_compare.to_csv(diff_path, index=False)
    print(f"\nDetail perbandingan disimpan di: {diff_path}")

except FileNotFoundError:
    print(f"File tidak ditemukan: {submission_lama_path}")
except Exception as e:
    print(f"Error saat membandingkan: {e}")